# Plate 05 · End-to-end Pipeline Evaluation (Kuwaiti plates)

Measures what the worker actually experiences:
**frame → detector + corners → warp → 4 cells → recogniser → Arabic/English cross-check → vote over best-3 frames**.

Targets (spec section 11): **95 % exact match by day, 90 % at night**, < 1.5 s on device.
Report separately: day / night / summer glare, plate type, distance, cross-check agreement rate,
and the share sent to the server fallback (confidence < 0.85).

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import time
from collections import Counter
import cv2, numpy as np, pandas as pd, torch
from ultralytics import YOLO
from Common_Utils.Data_Utils import variance_of_laplacian
from Common_Utils.Metrics_Utils import exact_match, character_error_rate
from Common_Utils.Plate_Normalizer import normalize, cross_check, is_valid
from Common_Utils.Kuwait_Plate_Layout import warp, split_cells

detector = YOLO(str(RUNS / "Plate_Detector" / "yolo11n_pose_kw_v1" / "weights" / "best.pt"))
CONF_THRESHOLD = 0.85

In [ ]:
def read_plate(frame, recogniser):
    """One frame -> (key "CODE-NUMBER", confidence, both rows agreed?) or (None, 0, False).
    recogniser(list_of_cell_images) -> (texts, confs)."""
    result = detector.predict(frame, imgsz=640, conf=0.4, verbose=False)[0]
    if result.keypoints is None or len(result.boxes) == 0:
        return None, 0.0, False
    corners = result.keypoints.xy[0].cpu().numpy()            # TL, TR, BR, BL
    cells = split_cells(warp(frame, corners))
    names = ["code_ar", "code_lat", "num_ar", "num_lat"]
    texts, confs = recogniser([cv2.cvtColor(cells[n], cv2.COLOR_BGR2GRAY) for n in names])
    code, c_conf, c_ok = cross_check(texts[0], texts[1], confs[0], confs[1])
    number, n_conf, n_ok = cross_check(texts[2], texts[3], confs[2], confs[3])
    if not is_valid(code, number):
        return None, 0.0, False
    return f"{code}-{number}", min(c_conf, n_conf), c_ok and n_ok

def recognise_sequence(frames, recogniser):
    """frames: BGR images sampled every 150 ms (spec 11.1). Keeps the 3 sharpest plate frames and votes."""
    scored = sorted(frames, key=lambda f: -variance_of_laplacian(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)))[:3]
    reads = [r for r in (read_plate(f, recogniser) for f in scored) if r[0]]
    if not reads:
        return None, 0.0, False
    winner, count = Counter(k for k, _, _ in reads).most_common(1)[0]
    agreeing = [r for r in reads if r[0] == winner]
    conf = float(np.mean([c for _, c, _ in agreeing])) * count / len(reads)
    return winner, conf, all(ok for _, _, ok in agreeing)

In [ ]:
# Evaluate on a held-out CSV: frames_folder, true_text, condition (day / night / glare), plate_type
EVAL_CSV = DATA / "Plates" / "Eval_Sequences.csv"
if EVAL_CSV.exists():
    df = pd.read_csv(EVAL_CSV)
    # df["pred"], df["conf"], df["agreed"], df["latency_ms"] = ...  (run recognise_sequence per row)
    for cond, part in df.groupby("condition"):
        print(cond, "exact:", exact_match(part["pred"].tolist(), part["true_text"].map(normalize).tolist()),
              "| rows agreed:", part["agreed"].mean() if "agreed" in part else "n/a")
else:
    print("Create", EVAL_CSV, "from consented Kuwaiti pilot-station recordings first.")